System construction and test


In [1]:
from datetime import date, datetime
import pandas as pd
import yfinance as yf
import time
import numpy as np
pd.options.mode.chained_assignment = None  # default='warn'


input:
    
     Titulo  : example: GGAL
     frequencia de tick : 1h (frequencia mais alta)
     dftitulo : vista do BD sqtitulosalpha.bd (testar ultimos periodas)
Parametros a variar para back testing ( definir rangos de variação)
     k, d, smooth (parametros do stch)
     dayM  (frequencia media) (multiplicador da frequencia mais alta) exemplo: 5
     semM  (frequencia baixa) (multiplicador da frequencia media) exemplo: 7

In [4]:
dataini = '2020-04-03 19:30:00'
datafim = '2024-04-03 19:30:00'

In [6]:
#%%timeit
import sqlite3
import pandas as pd

# Caminho para o banco de dados
caminho_bd = r'C:\Users\scitr\anaconda_projects\Trading_System\Dados_Fontes\Alpha_Vantage\sqtitulosalpha.db'

# Conectando ao banco
conexao = sqlite3.connect(caminho_bd)

# Lendo a view
#consulta = 'SELECT * FROM vwtitulosdados ORDER BY datetime'
consulta = f"""
SELECT * FROM vwtitulosdados
WHERE datetime BETWEEN '{dataini}' AND '{datafim}'
ORDER BY datetime
"""

dftitulosdados = pd.read_sql_query(consulta, conexao)

# Fechando a conexão
conexao.close()

# Exibindo os primeiros registros para conferir
#display(dftitulosdados)
dftitulosdados = dftitulosdados.drop(columns=["symbol", "moeda", "intervalo","volume"])
display(len(dftitulosdados))
#display(dftitulosdados.head(10))

8957

In [264]:
dfmetricas = None

In [328]:
# Parameters

i = 'high'
K = 9
D = 6   
smoth = 3 
medM = 5 
lowM = 5
stpl = 0.025
comission = 0.0035
taxalivrerisgoprom = 0.05
drawmax = -0.1


Trading System

In [331]:
#%%timeit
# Trading System

# Stochastic calculation
def stochastic(dftitulosdados, i, K, D, smoth):
    df = dftitulosdados    
    df["k"] = (100. * (df.close - df.low.rolling(K).min()) /
        (df.high.rolling(K).max() - df.low.rolling(K).min()))
    
    df["k" + i ] = df.k.rolling(smoth).mean()
    df["d" + i ] = df["k" + i].rolling(D).mean()
    
    df.drop(columns=["k"], inplace=True)  
    dfstoch = df
    return dfstoch
    
dfstoch = stochastic(dftitulosdados, i, K, D, smoth)

# stochastic high, med and low frcuency
def stoch_hml( dfstoch, k, d, smth, medM, lowM): 
    df = dfstoch
    df = stochastic(df, "high", k, d, smth)
    df = stochastic(df, "med", k*medM, d*medM, smth*medM)
    df = stochastic(df, "low", k*medM*lowM, d*medM*lowM, smth*medM*lowM)
    dfstoch_hml = df
    return dfstoch_hml
    
dfstoch_hml= stoch_hml(dfstoch , K, D, smoth, medM , lowM )

# CRiterias calculation
def system_criterias (dfstoch_hml):   # input  df() =  dfstoch_hml ()
    df = dfstoch_hml
    df["longbuylow"] = ((df["klow"] > 20) & (df["klow"] > df["dlow"])).astype(int)
    df["longbuymed"] = ((df["kmed"] > 20) & (df["kmed"] > df["dmed"])).astype(int)
    df["longbuyhigh"] = ((df["khigh"] > 20) & (df["khigh"] > df["dhigh"])).astype(int)
    dfcriterias = df
    return dfcriterias
    
dfcriterias = system_criterias (dfstoch_hml) 

# Signals "Enter" , "Out" , Standby" calculation
def system_signals (dfcriterias):
    df = dfcriterias
    n = len(df)
    state_array = np.full(n, "standby", dtype=object)  # inicializa com "standby"
    estado_anterior = "standby"

    high = df["longbuyhigh"].to_numpy()
    med = df["longbuymed"].to_numpy()
    low = df["longbuylow"].to_numpy()

    for i in range(1, n):
        if high[i] == 1 and med[i] == 1 and low[i] == 1 and estado_anterior == "standby":
            state_array[i] = "enter"
            estado_anterior = "enter"
        elif med[i] == 1 and low[i] == 1 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif high[i] == 1 and low[i] == 1 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "stay"
            estado_anterior = "stay"
        elif low[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif high[i] == 0 and med[i] == 0 and estado_anterior in ["enter", "stay"]:
            state_array[i] = "out"
            estado_anterior = "out"
        elif (low[i] == 0 or med[i] == 0) and estado_anterior == "out":
            state_array[i] = "standby"
            estado_anterior = "standby"
        else:
            state_array[i] = estado_anterior

    df["state"] = state_array
    dfsignals = df
    return df

dfsignals = system_signals (dfcriterias)




display (dfstoch.head(10))
display(dfstoch_hml.head(10) )
display (dfcriterias.head(10))
display (dfsignals.head(10))

Trade System calculations
Stop Loss , reentry
Index , Trade, Index with comissions
Stop System , Drawdowns > drawmax

In [335]:
#%%timeit
def stop_loss_reentry (dfsignals, stpl) :

    df = dfsignals[(dfsignals['state'] == 'enter') | (dfsignals['state'] == 'stay')]
    df = df.reset_index(drop=True)
    df = df.drop(columns=["open" ,	"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    #drop(columns=["symbol", "moeda", "intervalo","volume"])
    df["stpl"] = 0.0
    stoplossprice = 0.0
    lastlongbuyprice = 0.0

    for i in range(0, len(df)):
        
        if df.loc[i, "state"] == "enter" :
           stoplossprice = df.loc[i, "close"]
           df.loc[i,"stpl"] = df.loc[i, "close"] - stoplossprice * (1 - stpl)
           
        if df.loc[i, "state"]== "stay" :
           df.loc[i, "stpl"] = df.loc[i, "close"] - stoplossprice * (1- stpl)
            
           if df.loc[i, "stpl"] < 0.0 :
                df.loc[i, "state"] = "out"
               
           if df.loc[i, "stpl"] > 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "enter"
               
           if df.loc[i, "stpl"] < 0.0 and   (df.loc[i-1, "state"] == "out" or df.loc[i-1, "state"] == "outstpl") :
                df.loc[i, "state"] = "outstpl"

    return df
dfstoploss = stop_loss_reentry (dfsignals, stpl)


display(dfstoploss.head(10))

In [338]:
#%%timeit

#Index_sc, Index, Trade calculation

# clean dfsignals "out" duplicates for StopLoss 
def index_dataframe (dfsignals, dfstoploss) :
    # elimino colunas de dfsignals e filtro por os valores "out"
    dfsignalsdrop = dfsignals.drop(columns=["open" ,"high" , "low" , "khigh","dhigh","kmed" ,"dmed","klow","dlow"])
    dfsignalsout = dfsignalsdrop[(dfsignalsdrop['state'] == 'out')]

    # elimino a culuna stpl de dfstoploss e filtro os valores enter e out 
    dfstoplossdrop = dfstoploss.drop(columns=["stpl"]) 
    dfstoplossenterout = dfstoplossdrop[(dfstoplossdrop['state'] == 'enter') | (dfstoplossdrop['state'] == 'out')]

    # concatenar os dois df para ter o total dos signals enter e out
    dfsignalsenterout = pd.concat([dfsignalsout, dfstoplossenterout], ignore_index=True)

    # Ordenar pelo datetime e resetear o index
    dfsignalsenterout["datetime"] = pd.to_datetime(dfsignalsenterout["datetime"])
    dfsignalsenterout = dfsignalsenterout.sort_values("datetime").reset_index(drop=True)

    # limpar os out duplicados"out" por a saida anticipada do stoploss e reiniciar indice
    df = dfsignalsenterout
    cond = (df["state"] == "out")  & (df["state"].shift(1) == "out")
    dfsignalsentoutclean = df[~cond].reset_index(drop=True)
    return dfsignalsentoutclean
    
dfsignalsentoutclean = index_dataframe (dfsignals, dfstoploss)

# Indexes calculation
def index_trade(dfsignalsenteroutclean):
    df = dfsignalsentoutclean
    df ["index_sc"] = 100. 
    df ["trade"] = 0.
    df ["index"] = 100. *(1-comission) 
    for i in range(1, len(df)):      
                       
        if  df.loc[i, "state"] == "out" :
            df.loc[i, "index_sc"] = (((df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"])+1)* df.loc[i-1,"index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
            df.loc[i, "trade"] = (df.loc[i,"close"]-df.loc[i-1,"close"])/df.loc[i-1,"close"]
            
        if  df.loc[i, "state"] == "enter" :        
            df.loc[i, "index_sc"] =  df.loc[i-1, "index_sc"]
            df.loc[i, "index"] = df.loc[i, "index_sc"]* (1-comission)
    dfindex = df
    return dfindex

dfindex = index_trade(dfsignalsentoutclean)



display (dfsignalsentoutclean.head(10))
display (dfindex.head(10))

stop drawdawn

In [342]:
#%%timeit
def stop_drawdawn (dfindex, drawmax):
    df = dfindex
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df["index"] = pd.to_numeric(df["index"], errors="coerce")

    estado_corrigido = []
    pico_atual = df.loc[0, "index"]

    for i in range(len(df)):
        valor_index = df.loc[i, "index"]
        estado = df.loc[i, "state"]

        # Atualiza pico se houve recuperação
        if valor_index > pico_atual:
            pico_atual = valor_index

        # Calcula drawdown
        if pico_atual > 0:
            drawdown = (valor_index - pico_atual) / pico_atual
        else:
            drawdown = 0

        # Verifica se deve aplicar stopsys
        if estado == "out" and drawdown < drawmax:
            estado = "stopsys"
            pico_atual = valor_index  # reinicia ciclo a partir desse ponto

        estado_corrigido.append(estado)
    df["state"] = estado_corrigido
    dfindexdrawdawn = df
    return dfindexdrawdawn

dfindexdrawdawn = stop_drawdawn(dfindex, drawmax)
    


display(dfindexdrawdawn)

Metricas

In [346]:
# creo dataframe para calculo de metricas
dfinputmetricas = dfindexdrawdawn[['datetime', 'state','index_sc', 'index','trade']]



display(dfinputmetricas.head())

,datetime,state,index_sc,index,trade
0,2020-07-13 07:00:00,enter,100.000000,99.650000,0.000000
1,2020-07-13 09:00:00,out,95.749760,95.414636,-0.042502
2,2020-07-13 10:00:00,enter,95.749760,95.414636,0.000000
3,2020-07-13 15:00:00,out,91.632132,91.311419,-0.043004
4,2020-07-22 12:00:00,enter,91.632132,91.311419,0.000000


Metricas Tir

Tir total anualizada

In [350]:
lsmetricas = []

In [352]:

def tir_total_anualizada(dfinputmetricas, lsmetricas):
    df = dfinputmetricas
    # Garante que datetime está no formato certo
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Filtra enter e out
    df_enter = df[df['state'] == 'enter']
    df_out = df[df['state'] == 'out']

    # Verificação
    if df_enter.empty or df_out.empty:
        return None

    # Índice inicial e final
    idx_inicio = df_enter.iloc[0]['index']
    idx_fim = df_out.iloc[-1]['index']

    # Período completo entre primeira e última data do DataFrame
    dt_inicio_total = df['datetime'].min()
    dt_fim_total = df['datetime'].max()
    dias_total = (dt_fim_total - dt_inicio_total).days

    # Validação
    if dias_total <= 0 or idx_inicio == 0:
        return None

    # TIR anualizada com base no período total do df
    tirtotalanual = (idx_fim / idx_inicio) ** (365 / dias_total) - 1
    setirtotalanual = pd.Series({'tirtotalanual': tirtotalanual})
    lsmetricas.append (setirtotalanual)
    return  setirtotalanual , lsmetricas
    

In [354]:
setirtotalanual , lsmetricas = tir_total_anualizada (dfinputmetricas, lsmetricas)
display (setirtotalanual)
display (lsmetricas)


tirtotalanual    0.224883
dtype: float64

[tirtotalanual    0.224883
 dtype: float64]

In [356]:
#%%timeit

# dataframe tir anuais calculation
def tir_anuais_df (dfinputmetricas, dataini, datafim):

    # Exemplo do DataFrame original
    df = dfinputmetricas
    dataini = pd.to_datetime(dataini)
    datafim = pd.to_datetime(datafim)
    
    # Lista para novos registros
    novos_registros = []
    
    # Verifica se dataini deve ser adicionado
    if dataini < df.iloc[0]['datetime']:
        novos_registros.append({
            'datetime': dataini,
            'state': '',
            'index': df.iloc[0]['index']
        })
    
    # Verifica se datafim deve ser adicionado
    if datafim > df.iloc[-1]['datetime']:
        novos_registros.append({
            'datetime': datafim,
            'state': '',
            'index': df.iloc[-1]['index']
        })
    
    # Adiciona os registros e ordena
    df = pd.concat([pd.DataFrame(novos_registros), df], ignore_index=True)
    df = df.sort_values(by='datetime').reset_index(drop=True)
    
    # Determina os anos, excluindo o último ano
    ano_inicial = df['datetime'].min().year
    ano_final = (df['datetime'].max().year)
    
    # Gera os anos do intervalo EXCLUINDO o último ano
    anos_validos = range(ano_inicial, ano_final-1 )  # << ajuste aqui
    
    # Lista para os novos registros
    novos_registros = []
    
    for ano in anos_validos:
        fim_do_ano = pd.to_datetime(f'{ano}-12-31 23:59:59')
        df_antes = df[df['datetime'] < fim_do_ano]
        if not df_antes.empty:
            index_valor = df_antes.iloc[-1]['index']
            novos_registros.append({
                'datetime': fim_do_ano,
                'state': '',
                'index': index_valor
            })
    
    # Adiciona e organiza
    df = pd.concat([df, pd.DataFrame(novos_registros)], ignore_index=True)
    df = df.sort_values('datetime').reset_index(drop=True)
    
    # calcula o dataframe com as tir anuales ao fim do ano , com os anos incompletos anualizadas
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    #  consolidar por dia e manter o último registro
    df['date'] = df['datetime'].dt.date
    df_diario = df.sort_values('datetime').groupby('date', as_index=False).last()
    
    #  selecionar datas de fim de ano
    df_fim_ano = df_diario[
        (pd.to_datetime(df_diario['date']).dt.month == 12) &
        (pd.to_datetime(df_diario['date']).dt.day == 31)
    ].copy()
    
    #  calcular TIR entre pares de fim de ano
    resultados = []
    
    for i in range(1, len(df_fim_ano)):
        dt_inicio = pd.to_datetime(df_fim_ano.iloc[i - 1]['date'])
        dt_fim = pd.to_datetime(df_fim_ano.iloc[i]['date'])
        idx_inicio = df_fim_ano.iloc[i - 1]['index']
        idx_fim = df_fim_ano.iloc[i]['index']
        dias = (dt_fim - dt_inicio).days
    
        if dias > 0 and idx_inicio != 0:
            tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
            resultados.append({
                'datetime': dt_fim,
                'tiranual': tir
            })
    
    #  adicionar último intervalo incompleto
    if not df_fim_ano.empty:
        dt_inicio = pd.to_datetime(df_fim_ano.iloc[-1]['date'])
        idx_inicio = df_fim_ano.iloc[-1]['index']
        dt_fim = pd.to_datetime(df_diario.iloc[-1]['date'])
        idx_fim = df_diario.iloc[-1]['index']
        dias = (dt_fim - dt_inicio).days
    
        if dias > 0 and idx_inicio != 0:
            tir = (idx_fim / idx_inicio) ** (365 / dias) - 1
            resultados.append({
                'datetime': dt_fim,
                'tiranual': tir
            })

    # criar DataFrame final
    dftiranual = pd.DataFrame(resultados)
    return dftiranual

dftiranual = tir_anuais_df (dfinputmetricas, dataini, datafim)

#estatistic calculation
def tir_anuais_estat(dftiranual, lsmetricas):
    tir = dftiranual['tiranual'].dropna()

    estatisticas = {
        'tiranualquant': tir.count(),
        'tiranualfirst': round(tir.iloc[0], 6),
        'tiranualmedia': round(tir.mean(), 6),
        'tiranualmax': round(tir.max(), 6),
        'tiranualmin': round(tir.min(), 6),
        'tiranualstd': round(tir.std(), 6)
    }
    setiranuaisestats = pd.Series(estatisticas)
    lsmetricas.append (setiranuaisestats)
    return setiranuaisestats , lsmetricas


In [358]:
setiranuaisestats , lsmetricas = tir_anuais_estat(dftiranual, lsmetricas)
display (dftiranual, lsmetricas)

,datetime,tiranual
0,2021-12-31,0.391811
1,2022-12-31,-0.061139
2,2024-04-03,0.393028


[tirtotalanual    0.224883
 dtype: float64,
 tiranualquant    3.000000
 tiranualfirst    0.391811
 tiranualmedia    0.241233
 tiranualmax      0.393028
 tiranualmin     -0.061139
 tiranualstd      0.261863
 dtype: float64]

Metrica Trades

Trades Estatísticas

In [362]:
# trades estatistics calculation
def trades_estatisticas(dfinputmetricas, lsmetricas):
    df = dfinputmetricas.copy()
    trades = df["trade"].dropna()

    positivos = trades[trades > 0]
    negativos = trades[trades < 0]

    # Porcentagem de positivos
    porcentagem_pos = (len(positivos) / len(trades)) if len(trades) > 0 else 0

    ditradesestat = {
        "tradestot": len(trades),
        'tradefirst': round(trades.iloc[1], 6),
        "tradespositpor": round(porcentagem_pos, 6),
        "tradesposmedia": round(positivos.mean(), 6) if not positivos.empty else None,
        "tradesposstd": round(positivos.std(), 6) if not positivos.empty else None,
        "tradesposmax": round(positivos.max(), 6) if not positivos.empty else None,
        "tradesposmin": round(positivos.min(), 6) if not positivos.empty else None
    }

    setradesestats = pd.Series(ditradesestat)
    lsmetricas.append (setradesestats)
    return setradesestats , lsmetricas

setradesestats, lsmetricas = trades_estatisticas(dfinputmetricas, lsmetricas)

In [364]:

display (lsmetricas)

[tirtotalanual    0.224883
 dtype: float64,
 tiranualquant    3.000000
 tiranualfirst    0.391811
 tiranualmedia    0.241233
 tiranualmax      0.393028
 tiranualmin     -0.061139
 tiranualstd      0.261863
 dtype: float64,
 tradestot         202.000000
 tradefirst         -0.042502
 tradespositpor      0.163366
 tradesposmedia      0.075038
 tradesposstd        0.069721
 tradesposmax        0.253870
 tradesposmin        0.001335
 dtype: float64]

Metrica Drawdown

Drawdown Dataframe

In [368]:
#Metrica Drawdown Calculation

#Dataframe Drawdown Calculation
def drawdowns_df (dfinputmetricas):
   
    df = dfinputmetricas
    df["datetime"] = pd.to_datetime(df["datetime"])
    serie = df["index"].dropna().reset_index(drop=True)
    datas = df["datetime"].reset_index(drop=True)

    drawdowns = []

    pico_idx = 0
    pico = serie[0]
    vale_idx = None
    valor_vale = None
    max_dd = 0

    for i in range(1, len(serie)):
        if serie[i] > pico:
            # Se recuperou acima do último pico: salvar ciclo anterior
            if vale_idx is not None and max_dd < 0:
                drawdowns.append({
                    "Data Pico": datas[pico_idx],
                    "Valor Pico": pico,
                    "Data Vale": datas[vale_idx],
                    "Valor Vale": valor_vale,
                    "Drawdown (%)": round(max_dd * 100, 2)
                })

            # Novo pico inicia novo ciclo
            pico = serie[i]
            pico_idx = i
            vale_idx = None
            max_dd = 0
        else:
            dd = (serie[i] - pico) / pico
            if dd < max_dd:
                max_dd = dd
                vale_idx = i
                valor_vale = serie[i]

    # Salva último ciclo, se aplicável
    if vale_idx is not None and max_dd < 0:
        drawdowns.append({
            "Data Pico": datas[pico_idx],
            "Valor Pico": pico,
            "Data Vale": datas[vale_idx],
            "Valor Vale": valor_vale,
            "Drawdown (%)": round(max_dd * 100, 2)
        })

    # Retorna os top N
    df_resultado = pd.DataFrame(drawdowns)
    return df_resultado.sort_values("Drawdown (%)").reset_index(drop=True)  

dfdrawdowns = drawdowns_df(dfinputmetricas)

# Drawdowns statictics calculation
def drawdowns_estat(dfdrawdowns , lsmetricas):
    dd = dfdrawdowns['Drawdown (%)'].dropna()  # Filtra nulos, se houver

    estatisticas = {
        'drawdfirst': round(dd.iloc[0], 6),
        'drawdtot': dd.count(),
        'drawdmedia': round(dd.mean(), 2),
        'drawdmaximo': round(dd.max(), 2),
        'drawdminimo': round(dd.min(), 2),
        'drawdstd': round(dd.std(), 2)
    }
    sedrawdownsestats = pd.Series(estatisticas)
    lsmetricas.append (sedrawdownsestats)
    return sedrawdownsestats , lsmetricas

sedrawdownsestats , lsmetricas = drawdowns_estat(dfdrawdowns, lsmetricas)


In [370]:
display(lsmetricas)
#display (dfdrawdowns)

[tirtotalanual    0.224883
 dtype: float64,
 tiranualquant    3.000000
 tiranualfirst    0.391811
 tiranualmedia    0.241233
 tiranualmax      0.393028
 tiranualmin     -0.061139
 tiranualstd      0.261863
 dtype: float64,
 tradestot         202.000000
 tradefirst         -0.042502
 tradespositpor      0.163366
 tradesposmedia      0.075038
 tradesposstd        0.069721
 tradesposmax        0.253870
 tradesposmin        0.001335
 dtype: float64,
 drawdfirst    -27.87
 drawdtot       11.00
 drawdmedia    -10.69
 drawdmaximo    -2.76
 drawdminimo   -27.87
 drawdstd        8.80
 dtype: float64]

Metrica Dias Out

Dias Out Dataframe

In [374]:
def dias_out_df (dfmetricas):
    
    df = dfmetricas
    coluna="index_sc"
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df[df[coluna].notna()].reset_index(drop=True)

    variacao = df[coluna].diff()
    grupos = (variacao != 0).cumsum()

    agrupado = df.groupby(grupos)
    periodos_estaticos = []

    for _, grupo in agrupado:
        if len(grupo) > 1 and grupo[coluna].nunique() == 1:
            duracao_dias = (grupo["datetime"].iloc[-1] - grupo["datetime"].iloc[0]).days
            periodos_estaticos.append({                
                "Data Início": grupo["datetime"].iloc[0],
                "Data Fim": grupo["datetime"].iloc[-1],
                "difdias": duracao_dias,
                "Valor index": grupo[coluna].iloc[0]
            })

    dfdiasout = pd.DataFrame(periodos_estaticos)
    dfdiasout = dfdiasout.query("difdias != 0").copy()
    dfdiasout = dfdiasout.reset_index(drop=True)
    
    return dfdiasout
    
dfdiasout = dias_out_df (dfinputmetricas)


In [376]:

display(dfdiasout.head())


,Data Início,Data Fim,difdias,Valor index
0,2020-07-13 15:00:00,2020-07-22 12:00:00,8,91.632132
1,2020-07-27 15:00:00,2020-08-03 12:00:00,6,95.064094
2,2020-08-05 15:00:00,2020-11-02 16:00:00,89,97.537334
3,2020-11-09 10:00:00,2020-11-17 13:00:00,8,115.860369
4,2020-11-19 13:00:00,2020-11-24 10:00:00,4,110.541256


Dias Out Estatisticas

In [379]:
def dias_out_estats(dfdiasout, lsmetricas):
    dias = dfdiasout['difdias'].dropna()  # Remove valores nulos, se houver

    estatisticas = {
        'diasoutfirst': round(dias.iloc[0], 6),
        'diasouttot': dias.sum(),
        'diasoutmedia': round(dias.mean(), 2),
        'diasoutmax': dias.max(),
        'diasoutmin': dias.min(),
        'diasoutstd': round(dias.std(), 2)
    }
    sediasoutestats = pd.Series(estatisticas)
    lsmetricas.append(sediasoutestats)    
    return sediasoutestats, lsmetricas

sediasoutestats, lsmetricas = dias_out_estats(dfdiasout, lsmetricas)

In [381]:

display(lsmetricas)

[tirtotalanual    0.224883
 dtype: float64,
 tiranualquant    3.000000
 tiranualfirst    0.391811
 tiranualmedia    0.241233
 tiranualmax      0.393028
 tiranualmin     -0.061139
 tiranualstd      0.261863
 dtype: float64,
 tradestot         202.000000
 tradefirst         -0.042502
 tradespositpor      0.163366
 tradesposmedia      0.075038
 tradesposstd        0.069721
 tradesposmax        0.253870
 tradesposmin        0.001335
 dtype: float64,
 drawdfirst    -27.87
 drawdtot       11.00
 drawdmedia    -10.69
 drawdmaximo    -2.76
 drawdminimo   -27.87
 drawdstd        8.80
 dtype: float64,
 diasoutfirst       8.00
 diasouttot      1074.00
 diasoutmedia      16.52
 diasoutmax        89.00
 diasoutmin         1.00
 diasoutstd        20.98
 dtype: float64]

Metrica StopSys Dataframe

In [384]:
def stopsys_df (dfinputmetricas) :   
    # Garante que a coluna 'datetime' esteja no formato correto
    dfinputmetricas['datetime'] = pd.to_datetime(dfinputmetricas['datetime'])
    
    # Filtra os registros onde state == 'stopsys'
    dfstopsys = dfinputmetricas[dfinputmetricas['state'] == 'stopsys'][['datetime', 'state', 'index']].copy()
    
    #  Ordena por datetime
    dfstopsys = dfstopsys.sort_values('datetime').reset_index(drop=True)
    return dfstopsys
    
dfstopsys = stopsys_df (dfinputmetricas)

def stopsys_estat(dfstopsys, lsmetricas):
    df = dfstopsys.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.sort_values('datetime').reset_index(drop=True)

    index_values = df['index'].dropna()

    estatisticas = {
        'stopsysfirst': round(index_values.iloc[0], 6),
        'stopsysquant': index_values.count(),
        'stopsysmedia': round(index_values.mean(), 6),
        'stopsysmaximo': round(index_values.max(), 6),
        'stopsysminimo': round(index_values.min(), 6),
        'stopsystd': round(index_values.std(), 6)
    }
    sestopsysestats = pd.Series(estatisticas)
    lsmetricas.append(sestopsysestats)
    return sestopsysestats, lsmetricas

sestopsysestats, lsmetricas = stopsys_estat(dfstopsys, lsmetricas)


In [386]:

display (lsmetricas )

[tirtotalanual    0.224883
 dtype: float64,
 tiranualquant    3.000000
 tiranualfirst    0.391811
 tiranualmedia    0.241233
 tiranualmax      0.393028
 tiranualmin     -0.061139
 tiranualstd      0.261863
 dtype: float64,
 tradestot         202.000000
 tradefirst         -0.042502
 tradespositpor      0.163366
 tradesposmedia      0.075038
 tradesposstd        0.069721
 tradesposmax        0.253870
 tradesposmin        0.001335
 dtype: float64,
 drawdfirst    -27.87
 drawdtot       11.00
 drawdmedia    -10.69
 drawdmaximo    -2.76
 drawdminimo   -27.87
 drawdstd        8.80
 dtype: float64,
 diasoutfirst       8.00
 diasouttot      1074.00
 diasoutmedia      16.52
 diasoutmax        89.00
 diasoutmin         1.00
 diasoutstd        20.98
 dtype: float64,
 stopsysfirst     102.906207
 stopsysquant       7.000000
 stopsysmedia     158.490835
 stopsysmaximo    241.898821
 stopsysminimo    102.906207
 stopsystd         52.251601
 dtype: float64]

In [388]:


def atualizar_df_metricas(dfmetricas, lsmetricas):
    """
    Adiciona uma linha ao DataFrame dfmetricas com os valores de lsmetricas.
    Se dfmetricas for None, cria o DataFrame com a estrutura das métricas.

    Parâmetros:
    - dfmetricas: pd.DataFrame ou None
    - lsmetricas: list de pd.Series

    Retorna:
    - pd.DataFrame atualizado
    """

    linha = pd.concat(lsmetricas)  # Une todas as Series em uma só

    if dfmetricas is None:
        # Cria o DataFrame com uma única linha
        dfmetricas = pd.DataFrame([linha.values], columns=linha.index)
    else:
        # Adiciona nova linha ao DataFrame existente
        dfmetricas.loc[len(dfmetricas)] = linha.values

    return dfmetricas

dfmetricas = atualizar_df_metricas(dfmetricas, lsmetricas)

In [390]:
display (dfmetricas)

,tirtotalanual,tiranualquant,tiranualfirst,tiranualmedia,tiranualmax,tiranualmin,tiranualstd,tradestot,tradefirst,tradespositpor,...,diasoutmedia,diasoutmax,diasoutmin,diasoutstd,stopsysfirst,stopsysquant,stopsysmedia,stopsysmaximo,stopsysminimo,stopsystd
0,0.183607,3.0,0.298816,0.246959,0.298816,0.150297,0.083785,226.0,-0.034572,0.163717,...,14.17,60.0,1.0,16.12,94.453358,7.0,164.628915,234.272683,94.453358,55.885133
1,0.224883,3.0,0.391811,0.241233,0.393028,-0.061139,0.261863,202.0,-0.042502,0.163366,...,16.52,89.0,1.0,20.98,102.906207,7.0,158.490835,241.898821,102.906207,52.251601
